# LoRA Fine-Tuning and Reward Model Training - Qwen2.5-1.5B-Instruct + Qwen2.5- 27B-Instruct PIPELINE

This notebook demonstrates a pipeline for fine-tuning a Large Language Model (LLM) using LoRA (PEFT) and training a reward model for claim verification. It includes steps for data preprocessing, data augmentation, model training, and prediction generation.

## 1. Setup and Environment
Mount Google Drive, install dependencies, and authenticate with Hugging Face.

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login

# Mount Google Drive to access files
drive.mount('/content/drive')

# Retrieve Hugging Face token from Colab secrets and log in
HF_TOKEN = userdata.get('login_key')
login(token=HF_TOKEN, add_to_git_credential=True)


Mounted at /content/drive


In [ ]:
# Install necessary libraries
!pip install -q transformers peft evaluate tomli camel-tools accelerate bitsandbytes>=0.46.1 torchao

import evaluate
print("Evaluate library version:", evaluate.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 29.8 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import time
import random
import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader

from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

from sklearn.metrics import accuracy_score
import evaluate

from peft import LoraConfig, get_peft_model

In [ ]:
torch.set_default_dtype(torch.float32)


## 2. Helper Functions and Model Classes
Define the text preprocessing functions, custom PyTorch dataset, and the core classifier logic incorporating LoRA.

In [ ]:
import evaluate
import datetime
import re

# Initialize the metric
f1_metric = evaluate.load("f1")

def format_time(elapsed):
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))

def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")

def print_trainable_parameters(model):
    trainable_params, all_param = 0, 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print(
        f"trainable params: {trainable_params} || "
        f"all params: {all_param} || "
        f"trainable%: {100 * trainable_params / all_param:.2f}"
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
class CustomClassifier(torch.nn.Module):
    def __init__(
        self,
        model_name,
        num_labels=1,
        hidden_dim=None,
        dropout_value=0.1,
        freeze_base_layer=True,
        use_lora=False,
        is_base_encoder=True,
        lora_rank=16,
        lora_alpha=32,
    ):
        super().__init__()

        self.model = AutoModel.from_pretrained(model_name)
        self.is_base_encoder = is_base_encoder

        if freeze_base_layer:
            for param in self.model.parameters():
                param.requires_grad = False

        if use_lora:
            lora_config = LoraConfig(
                r=lora_rank,
                lora_alpha=lora_alpha,
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                lora_dropout=0.05,
                bias="none",
                task_type="FEATURE_EXTRACTION"
            )
            self.model = get_peft_model(self.model, lora_config)

        hidden_size = self.model.config.hidden_size

        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(hidden_dim, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)

        if self.is_base_encoder:
            pooled_output = outputs.pooler_output
        else:
            # Use mean pooling or CLS token if pooler is not available
            pooled_output = outputs.last_hidden_state[:, -1, :]

        pooled_output = pooled_output.to(torch.device("cuda"), dtype=torch.float)
        logits = self.classifier(pooled_output)
        return logits

In [ ]:
# This cell is now redundant as pip installs are consolidated. Keeping it empty for now.

In [ ]:
from camel_tools.utils.normalize import normalize_unicode, normalize_alef_maksura_ar, normalize_alef_ar, normalize_teh_marbuta_ar
from camel_tools.utils.dediac import dediac_ar

def camel_preprocess_arabic(text):
    if not isinstance(text, str):
        return text
    # Remove diacritics (Tashkeel)
    text = dediac_ar(text)
    # Normalize Unicode
    text = normalize_unicode(text)
    # Normalize Alef
    text = normalize_alef_ar(text)
    # Normalize Alef Maksura
    text = normalize_alef_maksura_ar(text)
    # Normalize Teh Marbuta
    text = normalize_teh_marbuta_ar(text)
    return text

In [ ]:
import os
import json

# Apply CAMeL preprocessing to the dataframes
train_df = pd.read_json(TRAIN_CSV, lines = True)
val_df = pd.read_json(VAL_CSV, lines = True)
with open(TEST_JSON, "r") as f:
  test_data = json.load(f)

train_df['input_text'] = train_df['input_text'].apply(camel_preprocess_arabic)
val_df['input_text'] = val_df['input_text'].apply(camel_preprocess_arabic)

# Helper function to apply CAMeL recursively to the test JSON
def deep_camel_preprocess_dict(obj):
    if isinstance(obj, dict):
        return {k: deep_camel_preprocess_dict(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [deep_camel_preprocess_dict(i) for i in obj]
    elif isinstance(obj, str):
        return camel_preprocess_arabic(obj)
    else:
        return obj

# Apply to the loaded test_data
test_data = deep_camel_preprocess_dict(test_data)

print("CAMeL preprocessing complete! Now your text is clean for the CheckThat! model.")


In [ ]:
# Define new paths for the fully preprocessed files
normalized_train_path = os.path.join("/content/drive/MyDrive/output", "Arabic_train_preprocessed.jsonl")
normalized_val_path = os.path.join("/content/drive/MyDrive/output", "Arabic_val_preprocessed.jsonl")
normalized_test_path = os.path.join("/content/drive/MyDrive/output", "Arabic_test_preprocessed.json")

# Save to Drive
train_df.to_json(normalized_train_path, orient="records", lines=True)
val_df.to_json(normalized_val_path, orient="records", lines=True)

with open(normalized_test_path, "w") as f:
    json.dump(test_data, f, indent=4, ensure_ascii=False)

print(f"Fully preprocessed train data saved to: {normalized_train_path}")
print(f"Fully preprocessed val data saved to: {normalized_val_path}")
print(f"Fully preprocessed test data saved to: {normalized_test_path}")

In [ ]:
import torch
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        # Include Questions in the input text if available
        if "Questions" in dataframe.columns:
            self.texts = dataframe.apply(lambda row: f"{row['input_text']}\nQuestions: {row['Questions']}", axis=1).tolist()
        else:
            self.texts = dataframe["input_text"].tolist()

        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

In [ ]:
class TrainerModule:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        epochs,
        lr,
        patience,
        output_dir,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs

        self.optimizer = AdamW(model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

        self.history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    def train(self, start_epoch=0):
        for epoch in range(start_epoch, self.epochs):
            print(f"\nEpoch {epoch+1}/{self.epochs}")
            self.model.train()

            total_loss, total_acc = 0, 0
            print_every = 100

            for batch_idx, batch in enumerate(tqdm(self.train_loader, desc=f"Training Epoch {epoch+1}")):
                self.optimizer.zero_grad()

                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)
                logits = self.model(input_ids, attention_mask)

                loss = self.loss_fn(logits.squeeze(1), labels)

                loss.backward()
                self.optimizer.step()
                self.scheduler.step()

                total_loss += loss.item()
                preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy().flatten()
                total_acc += accuracy_score(labels.cpu().numpy(), (preds >= 0.5).astype(int))

                if (batch_idx + 1) % print_every == 0:
                    print(f"Batch {batch_idx + 1}/{len(self.train_loader)} | "
                          f"Running Loss: {total_loss / (batch_idx + 1):.4f} | "
                          f"Running Acc: {total_acc / (batch_idx + 1):.4f}")

            avg_train_loss = total_loss / len(self.train_loader)
            avg_train_acc = total_acc / len(self.train_loader)
            self.history['train_loss'].append(avg_train_loss)
            self.history['train_acc'].append(avg_train_acc)

            print(f"End Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.4f}")
            self.evaluate(epoch)

    def evaluate(self, epoch):
        self.model.eval()
        total_loss, total_acc = 0, 0

        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Evaluating"):
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask)
                loss = self.loss_fn(logits.squeeze(1), labels)

                total_loss += loss.item()
                preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy().flatten()
                total_acc += accuracy_score(labels.cpu().numpy(), (preds >= 0.5).astype(int))

        avg_val_loss = total_loss / len(self.val_loader)
        avg_val_acc = total_acc / len(self.val_loader)
        self.history['val_loss'].append(avg_val_loss)
        self.history['val_acc'].append(avg_val_acc)

        print(f"Val Loss: {avg_val_loss:.4f} | Val Acc: {avg_val_acc:.4f}")

        # Save full checkpoint
        # Save the full model, optimizer, scheduler state and history
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'history': self.history
        }
        torch.save(checkpoint, os.path.join(self.output_dir, f"checkpoint_epoch_{epoch}.pt"))
        tokenizer.save_pretrained(self.output_dir)
        print(f"Checkpoint saved to {self.output_dir}")


In [ ]:
class VerifierEvaluator:
    def __init__(
        self,
        model_path,
        tokenizer_path,
        base_model,
        use_decomp,
        device="cuda",
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.use_decomp = use_decomp

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = CustomClassifier(base_model, use_lora=True,is_base_encoder=False,)

        # Load the full checkpoint dictionary
        checkpoint = torch.load(model_path, map_location=self.device)

        # Extract only the model's state_dict
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.to(self.device)
        self.model.eval()

    def encode_input(self, claim, questions, verdict, justification, max_length=150):

        # Added questions to the formatted text
        text = f"Claim: {claim}\nQuestions: {questions}\nVerdict: {verdict}\nJustification: {justification}"

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, questions, verdict, justification):
        ids, mask = self.encode_input(claim, questions, verdict, justification)
        with torch.no_grad():
            return float(torch.sigmoid(self.model(ids, mask)).item())

## 3. Global Configuration
Set up all paths, hyperparameter values, and model choices used across the notebook.

In [ ]:
# ===== GLOBAL CONFIGURATION AND PATHS =====

# Base models for fine-tuning and question generation
BASE_MODEL_TRAINING = "Qwen/Qwen2.5-1.5B-Instruct" # Model for LoRA fine-tuning
QWEN_QUESTION_MODEL = "Qwen/Qwen2.5-32B-Instruct" # Larger model for dynamic question generation

# Root directory for outputs in Google Drive
OUTPUT_ROOT_DIR = "/content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models"

# Raw data paths (assuming they are in a specific 'output' folder in Drive)
RAW_DATA_PATH = "/content/drive/MyDrive/output"
RAW_TRAIN_JSONL = f"{RAW_DATA_PATH}/Arabic_train_dynamic.jsonl"
RAW_VAL_JSONL   = f"{RAW_DATA_PATH}/Arabic_val_dynamic.jsonl"
RAW_TEST_JSON   = f"{RAW_DATA_PATH}/Arabic_test_dynamic.json"

# Preprocessed data paths (after CAMeL normalization)
CAMEL_TRAIN_JSONL = f"{OUTPUT_ROOT_DIR}/Arabic_train_preprocessed.jsonl"
CAMEL_VAL_JSONL   = f"{OUTPUT_ROOT_DIR}/Arabic_val_preprocessed.jsonl"
CAMEL_TEST_JSON   = f"{OUTPUT_ROOT_DIR}/Arabic_test_preprocessed.json"

# Data augmented with static questions
STATIC_AUG_TRAIN_JSONL = f"{OUTPUT_ROOT_DIR}/Arabic_train_static_augmented.jsonl"
STATIC_AUG_VAL_JSONL   = f"{OUTPUT_ROOT_DIR}/Arabic_val_static_augmented.jsonl"
STATIC_AUG_TEST_JSON   = f"{OUTPUT_ROOT_DIR}/Arabic_test_static_augmented.json"

# Data augmented with dynamic questions from Qwen
DYNAMIC_AUG_TRAIN_JSONL = f"{OUTPUT_ROOT_DIR}/Arabic_train_dynamic_qwen.jsonl"
DYNAMIC_AUG_VAL_JSONL   = f"{OUTPUT_ROOT_DIR}/Arabic_val_dynamic_qwen.jsonl"
DYNAMIC_AUG_TEST_JSON   = f"{OUTPUT_ROOT_DIR}/Arabic_test_dynamic_qwen.json"

# Paths for the final training data (choose between raw, camel, static, or dynamic)
# Initially, we'll point to camel-preprocessed. These will be updated later if augmentation is applied.
TRAIN_DATA_PATH = CAMEL_TRAIN_JSONL
VAL_DATA_PATH   = CAMEL_VAL_JSONL
TEST_DATA_PATH  = CAMEL_TEST_JSON

# Training Hyperparameters
MAX_LENGTH = 512
BATCH_SIZE = 4
EPOCHS = 3
LR = 1e-4

# Directory for saving model checkpoints and tokenizer
MODEL_OUTPUT_DIR = f"{OUTPUT_ROOT_DIR}/reward_model_checkpoints"


## 4. Exploratory Data Analysis (EDA)
Analyze sequence lengths and the distribution of digits in the Arabic text to better understand the dataset.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def get_token_lengths(dataframe, tokenizer):
    lengths = []
    for text in dataframe['input_text']:
        tokens = tokenizer.encode(text, add_special_tokens=True)
        lengths.append(len(tokens))
    return lengths

train_lengths = get_token_lengths(train_df, tokenizer)
val_lengths = get_token_lengths(val_df, tokenizer)

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.histplot(train_lengths, color='blue', label='Train Data', kde=True, stat='density', alpha=0.5)
sns.histplot(val_lengths, color='red', label='Validation Data', kde=True, stat='density', alpha=0.5)
plt.title('Distribution of Token Lengths')
plt.xlabel('Token Length')
plt.ylabel('Density')
plt.legend()
plt.show()

print(f"Train Data:")
print(f"  Mean token length: {np.mean(train_lengths):.2f}")
print(f"  Median token length: {np.median(train_lengths)}")
print(f"  90th percentile token length: {np.percentile(train_lengths, 90)}")
print(f"  Max token length: {np.max(train_lengths)}")

print(f"\nValidation Data:")
print(f"  Mean token length: {np.mean(val_lengths):.2f}")
print(f"  Median token length: {np.median(val_lengths)}")
print(f"  90th percentile token length: {np.percentile(val_lengths, 90)}")
print(f"  Max token length: {np.max(val_lengths)}")

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-1B.
401 Client Error. (Request ID: Root=1-69f8d2dc-58b890894488c29665583f9c;abc9005a-835f-4c83-b921-2158d745cc6b)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-1B/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-1B is restricted. You must have access to it and be authenticated to access it. Please log in.

## Exploratory Data Analysis: Digit Analysis

In [ ]:
# Define Western and Eastern Arabic digits
western_arabic_digits = set("0123456789")
eastern_arabic_digits = set("٠١٢٣٤٥٦٧٨٩")
train_df = pd.read_json(TRAIN_CSV, lines = True)
val_df = pd.read_json(VAL_CSV, lines = True)

def count_arabic_digits(text):
    western_count = 0
    eastern_count = 0
    for char in text:
        if char in western_arabic_digits:
            western_count += 1
        elif char in eastern_arabic_digits:
            eastern_count += 1
    return western_count, eastern_count

# Apply the function to the 'input_text' column of train_df
train_df[['western_digits_count', 'eastern_digits_count']] = train_df['input_text'].apply(lambda x: pd.Series(count_arabic_digits(x)))

# Apply the function to the 'input_text' column of val_df
val_df[['western_digits_count', 'eastern_digits_count']] = val_df['input_text'].apply(lambda x: pd.Series(count_arabic_digits(x)))

print("Train DataFrame Digit Counts Summary:")
display(train_df[['western_digits_count', 'eastern_digits_count']].describe())

print("\nValidation DataFrame Digit Counts Summary:")
display(val_df[['western_digits_count', 'eastern_digits_count']].describe())

Train DataFrame Digit Counts Summary:


,western_digits_count,eastern_digits_count
count,10105.000000,10105.000000
mean,11.509847,0.121920
std,9.455491,0.952234
min,0.000000,0.000000
25%,5.000000,0.000000
50%,10.000000,0.000000
75%,16.000000,0.000000
max,87.000000,20.000000



Validation DataFrame Digit Counts Summary:


,western_digits_count,eastern_digits_count
count,2473.000000,2473.000000
mean,12.056611,0.164173
std,10.148494,1.222653
min,0.000000,0.000000
25%,6.000000,0.000000
50%,10.000000,0.000000
75%,16.000000,0.000000
max,91.000000,32.000000


In [ ]:
import json

# Ensure test_data is loaded
with open(TEST_JSON, "r") as f:
    test_data = json.load(f)

# Convert test_data (list of dictionaries) to a pandas DataFrame
test_df = pd.DataFrame(test_data)

# Create 'input_text' column for test_df by combining relevant fields
# For simplicity in EDA, we'll take the first verdict and justification
test_df['input_text'] = test_df.apply(
    lambda row: f"Claim: {row['claim']}\nVerdict: {row['Verdict_list'][0]}\nJustification: {remove_label_pattern(row['Reasoning_traces'][0])}",
    axis=1
)

# Apply the function to the 'input_text' column of test_df
test_df[['western_digits_count', 'eastern_digits_count']] = test_df['input_text'].apply(lambda x: pd.Series(count_arabic_digits(x)))

print("Test DataFrame Digit Counts Summary:")
display(test_df[['western_digits_count', 'eastern_digits_count']].describe())

Test DataFrame Digit Counts Summary:


,western_digits_count,eastern_digits_count
count,511.000000,511.000000
mean,8.262231,0.023483
std,5.520880,0.305884
min,0.000000,0.000000
25%,4.000000,0.000000
50%,8.000000,0.000000
75%,11.000000,0.000000
max,41.000000,6.000000


## 5. Model Training
Initialize the model, set up the trainer, and begin training (or resume from a checkpoint).

In [ ]:
import torch
import os

# Define the checkpoint to load
checkpoint_path = os.path.join(OUTPUT_DIR, 'model_epoch_0.pt')

if 'model' not in locals() or 'trainer' not in locals():
    print("ERROR: 'model' or 'trainer' is not defined. Please run cell 39fd019f first.")
elif os.path.exists(checkpoint_path):
    print(f"Loading checkpoint: {checkpoint_path}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Check if the checkpoint file is a directory (meaning it's a saved model from save_pretrained)
    # or a .pt file (meaning it's a full checkpoint dict)
    if os.path.isdir(checkpoint_path): # This path will typically be a directory if saving a PEFT model
        # For PEFT models saved with save_pretrained, load the adapter weights
        # The base model should already be loaded in `model` instance
        # This part might need adjustment based on how the model was saved
        # For now, let's assume `model_epoch_0.pt` refers to a full state dict
        print(f"Warning: '{checkpoint_path}' is a directory. Attempting to load model_state_dict from a .pt file instead.")
        # Attempt to find the actual .pt file, assuming a standard naming convention
        actual_checkpoint_file = os.path.join(OUTPUT_DIR, 'checkpoint_epoch_0.pt') # Assuming `checkpoint_epoch_0.pt` exists
        if os.path.exists(actual_checkpoint_file):
            checkpoint = torch.load(actual_checkpoint_file, map_location=device)
        else:
            print(f"ERROR: Full checkpoint file '{actual_checkpoint_file}' not found.")
            checkpoint = None
    else: # It's a .pt file
        checkpoint = torch.load(checkpoint_path, map_location=device)

    if checkpoint and isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        trainer.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        trainer.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        trainer.history = checkpoint.get('history', trainer.history)
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resuming from epoch {start_epoch}")
    else:
        # Fallback: assume the file is just the state_dict
        print("Format mismatch or checkpoint not found: Loading file as raw state_dict.")
        # This line assumes checkpoint_path directly points to the model's state_dict
        # If it was a directory from save_pretrained, this would fail.
        # For now, assume it's a .pt file with just the state_dict.
        if checkpoint is not None: # Only attempt if checkpoint was successfully loaded as .pt
            model.load_state_dict(checkpoint)
        start_epoch = 1

    # Continue training from the determined start_epoch
    trainer.train(start_epoch=start_epoch)
else:
    print(f"No checkpoint found at: {checkpoint_path}")
    # Start training from scratch if no checkpoint is found
    trainer.train()

No checkpoint found at: /content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models/model_epoch_0.pt

Epoch 1/3


Training Epoch 1:   1%|          | 13/2527 [00:03<11:46,  3.56it/s]


KeyboardInterrupt: 

In [ ]:
from huggingface_hub import login
from google.colab import userdata
# Ensure you have run the cells defining TextDataset, CustomClassifier, and TrainerModule first
# Use add_to_git_credential=True to ensure the token is correctly configured for the environment
login(userdata.get('login_key'))
train_df = pd.read_json(TRAIN_CSV, lines = True)
val_df = pd.read_json(VAL_CSV, lines = True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

train_dataset = TextDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = TextDataset(val_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

lora_rank_value = 16
lora_alpha_value = 32

model = CustomClassifier(
    BASE_MODEL,
    use_lora=True,
    is_base_encoder=False,
    lora_rank=lora_rank_value,
    lora_alpha=lora_alpha_value,
)

print_trainable_parameters(model)

trainer = TrainerModule(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LR,
    patience=2,
    output_dir=OUTPUT_DIR,
)

# Note: You can comment out trainer.train() if you only want to initialize and then load a checkpoint
trainer.train()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Batch 2500/2527 | Running Loss: 0.0009 | Running Acc: 0.9998


Training Epoch 3: 100%|██████████| 2527/2527 [10:33<00:00,  3.99it/s]


End Epoch 3 | Train Loss: 0.0009 | Train Acc: 0.9998


Evaluating: 100%|██████████| 619/619 [00:56<00:00, 11.04it/s]


Val Loss: 0.1251 | Val Acc: 0.9742
Checkpoint saved to /content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models


OUTPUT_DIR check

In [ ]:
import os
if os.path.exists(OUTPUT_DIR):
    print(f"Files in {OUTPUT_DIR}:")
    print(os.listdir(OUTPUT_DIR))
else:
    print(f"The directory {OUTPUT_DIR} does not exist. Please check if your Google Drive is mounted and the path is correct.")

Files in /content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models:
['chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'model_epoch_0.pt']


## 6. Inference and Evaluation
Use the trained model (checkpoint) to score predictions on the test dataset and output the results.

In [ ]:
with open(TEST_JSON, "r") as f:
  test_data = json.load(f)

predictions = []

evaluator = VerifierEvaluator(
    model_path=f"{OUTPUT_DIR}/checkpoint_epoch_0.pt",
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    use_decomp=True,
)

# Example scoring
for idx, sample in enumerate(test_data):
  verdict_list = []
  verifier_score_list = []
  justification_list = []
  approved_majority_list = []
  for decoding_sample in range(1, len(sample["Reasoning_traces"]) + 1):
    justification = (
        remove_label_pattern(
            sample["Reasoning_traces"][decoding_sample - 1]
        ).split("Label:")[0]
    )
    score = evaluator.score(
    sample["claim"],
    sample.get("Questions", ""),
    sample["Verdict_list"][decoding_sample - 1].lower(),
    justification,
)
    verdict_list.append(sample["Verdict_list"][decoding_sample - 1])
    justification_list.append(justification)
    verifier_score_list.append(score)
    best_verdict = verdict_list[np.argmax(np.array(verifier_score_list))]
  predictions.append({
      "query_id": idx,
      "Claim": sample["claim"],
      "label": sample.get("label", best_verdict),
      "Verdict_BoN": best_verdict,
      "BoN_Verdict_list": verdict_list,
      "Reasoning_traces": justification_list,
      "score_list": verifier_score_list,
  })


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
len(test_data)

1600

In [ ]:
import os

output_file_path = f"/content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models/Data/Arabic/clef_predictions_final.json"
output_dir = os.path.dirname(output_file_path)

# Create the directory if it does not exist
os.makedirs(output_dir, exist_ok=True)

with open(output_file_path, "w") as fp:
  json.dump(predictions, fp, indent=4)

## 7. Data Augmentation
Enhance the training data by adding static or dynamically generated reasoning questions (via a larger LLM) to improve model robustness.

In [ ]:
import json
import pandas as pd
import os

# Define the 3 general questions
general_questions = "1. Evidence: Does the justification cite specific sources for all key entities?\n2. Alignment: Do dates, locations, and values match the claim exactly?\n3. Logic: Does the evidence provided fully necessitate the final verdict?"

def extend_dataset_with_static_questions(input_path, output_path, file_type='jsonl'):
    print(f"Processing {input_path}...")

    # Load data
    if file_type == 'jsonl':
        with open(input_path, 'r', encoding='utf-8') as f:
            data = [json.loads(line) for line in f]
    else: # json
        with open(input_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

    # Add questions
    for item in data:
        item['Questions'] = general_questions

    # Ensure output directory exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Save updated data
    if file_type == 'jsonl':
        with open(output_path, 'w', encoding='utf-8') as f:
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
    else:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"Saved extended dataset to {output_path}\n")
    return data

# Define output paths using OUTPUT_DIR
TRAIN_EXTENDED_CSV = f"{OUTPUT_DIR}/Arabic_train_extended.jsonl"
VAL_EXTENDED_CSV = f"{OUTPUT_DIR}/Arabic_val_extended.jsonl"
TEST_EXTENDED_JSON = f"{OUTPUT_DIR}/Arabic_test_extended.json"

# Run on all data
print("--- Extending datasets with static questions ---")
train_data_extended = extend_dataset_with_static_questions(TRAIN_CSV, TRAIN_EXTENDED_CSV, file_type='jsonl')
val_data_extended = extend_dataset_with_static_questions(VAL_CSV, VAL_EXTENDED_CSV, file_type='jsonl')
test_data_extended = extend_dataset_with_static_questions(TEST_JSON, TEST_EXTENDED_JSON, file_type='json')

# Show a sample to verify
print("Sample from validation set:")
print(json.dumps(val_data_extended[0], indent=2, ensure_ascii=False))

--- Extending datasets with static questions ---
Processing /content/drive/MyDrive/output/Arabic_train_preprocessed.jsonl...
Saved extended dataset to /content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models/Arabic_train_extended.jsonl

Processing /content/drive/MyDrive/output/Arabic_val_preprocessed.jsonl...
Saved extended dataset to /content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models/Arabic_val_extended.jsonl

Processing /content/drive/MyDrive/output/Arabic_test_preprocessed.json...
Saved extended dataset to /content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models/Arabic_test_extended.json

Sample from validation set:
{
  "sample_id": "row0_a",
  "input_text": "Claim: ناسا ترصد اكثر من 30 نجم في السماء الدنيا سرعتهم اكبر من سرعه الصوت\nVerdict: false\nJustification: الادعاء يقول ان وكاله ناسا ترصد اكثر من 30 نجم في السماء الدنيا سرعتهم اكبر من سرعه الصوت. لكن الادله المقدمه لا تشير الي اي معلومات تتعلق بنجوم او سرعتها، بل تتحدث عن كويكب يتج

In [ ]:
import torch
import json
import os
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import userdata

# Retrieve HF token from Colab secrets
hf_token = userdata.get('login_key')

# 1. Load the model locally in 4-bit to fit in Colab GPU
model_id = "Qwen/Qwen2.5-32B-Instruct"

print("Loading model and tokenizer...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
# Required for batching
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    token=hf_token
)
print("Model loaded successfully!")

# 2. Define the batch generation function
def generate_local_questions_batch(claims, verdicts):
    prompts = []
    for claim, verdict in zip(claims, verdicts):
        messages = [
            {"role": "system", "content": "You are an expert Fact-Check Decomposer. Break the claim down into exactly 3 atomic, verifiable questions, 1 numerical question, 1 temporal question and 1 logical question. Return ONLY the 3 questions, numbered 1 to 3."},
            {"role": "user", "content": f'Claim: "{claim}"\nVerdict: "{verdict}"'}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(text)

    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

    # Generate output
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=True
        )

    responses = []
    for i, output in enumerate(outputs):
        generated_ids = output[len(inputs.input_ids[i]):]
        response = tokenizer.decode(generated_ids, skip_special_tokens=True)
        responses.append(response.strip())

    return responses

# 3. Process the dataset in batches
def process_local(input_path, output_path, file_type='jsonl', batch_size=128):
    print(f"\nProcessing {input_path} in batches of {batch_size}...")

    if file_type == 'jsonl':
        with open(input_path, 'r', encoding='utf-8') as f:
            data = [json.loads(line) for line in f]
    else:
        with open(input_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

    with tqdm(total=len(data), desc="Generating Questions") as pbar:
        for i in range(0, len(data), batch_size):
            batch = data[i:i+batch_size]
            claims = []
            verdicts = []

            for item in batch:
                text = item.get('input_text', item.get('claim', ''))
                claim = text.split('\n')[0].replace('Claim: ', '').strip()
                verdict = item.get('Label', item.get('Verdict', ''))
                claims.append(claim)
                verdicts.append(verdict)

            # Generate questions for the whole batch
            questions_batch = generate_local_questions_batch(claims, verdicts)

            for item, questions in zip(batch, questions_batch):
                item['Questions'] = questions

            pbar.update(len(batch))

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    if file_type == 'jsonl':
        with open(output_path, 'w', encoding='utf-8') as f:
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
    else:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"\nSaved to {output_path}")

OUTPUT_DIR = "/content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models"
TRAIN_DYNAMIC_CSV = f"{OUTPUT_DIR}/Arabic_train_dynamic_qwen.jsonl"
VAL_DYNAMIC_CSV = f"{OUTPUT_DIR}/Arabic_val_dynamic_qwen.jsonl"
TEST_DYNAMIC_JSON = f"{OUTPUT_DIR}/Arabic_test_dynamic_qwen.json"

TRAIN_CSV = "/content/drive/MyDrive/output/Arabic_train_preprocessed.jsonl"
VAL_CSV   = "/content/drive/MyDrive/output/Arabic_val_preprocessed.jsonl"
TEST_JSON = "/content/drive/MyDrive/output/Arabic_test_preprocessed.json"

# Run on Validation Set as a test (Uncomment others when ready)
# process_local(VAL_CSV, VAL_DYNAMIC_CSV, file_type='jsonl', batch_size=128)
# process_local(TEST_JSON, TEST_DYNAMIC_JSON, file_type='json', batch_size=128)
# process_local(TRAIN_CSV, TRAIN_DYNAMIC_CSV, file_type='jsonl', batch_size=128)


Loading model and tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!

Processing /content/drive/MyDrive/output/Arabic_val_preprocessed.jsonl in batches of 64...


Generating Questions: 100%|██████████| 39/39 [28:09<00:00, 43.31s/it]


Saved to /content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models/Arabic_val_dynamic_qwen.jsonl


In [ ]:
process_local(TEST_JSON, TEST_DYNAMIC_JSON, file_type='json', batch_size=128)
process_local(TRAIN_CSV, TRAIN_DYNAMIC_CSV, file_type='jsonl', batch_size=128)
process_local(VAL_CSV, VAL_DYNAMIC_CSV, file_type='jsonl', batch_size=128)



Processing /content/drive/MyDrive/output/Arabic_test_preprocessed.json in batches of 128...


Generating Questions: 100%|██████████| 4/4 [04:23<00:00, 65.78s/it]



Saved to /content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models/Arabic_test_dynamic_qwen.json

Processing /content/drive/MyDrive/output/Arabic_train_preprocessed.jsonl in batches of 128...


Generating Questions: 100%|██████████| 79/79 [1:18:05<00:00, 59.31s/it]



Saved to /content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models/Arabic_train_dynamic_qwen.jsonl
